# Мини-исследование: текст как локальный сдвиг активаций LLM

Цель этого ноутбука - проверить не метафору напрямую, а измеримую версию гипотезы:

> Класс текстов `T` создает воспроизводимое направление в hidden states / residual stream модели, и это направление связано со сдвигом вероятностей следующих ответов.

Что проверяем:

1. **Hidden states**: отличаются ли внутренние представления после целевых текстов от нейтральных контролей.
2. **Logit shift**: меняются ли вероятности заданных вариантов ответа после таких текстов.
3. **Activation steering smoke test**: можно ли усилить или ослабить эффект, добавив контрастный вектор на выбранном слое.

Важно: один текст - это только демонстрация. Для результата, который можно показывать исследователям, нужно хотя бы 20-50 целевых текстов и столько же контрольных текстов похожей длины и языка.

In [ ]:
!pip -q install -U "transformers>=4.45.0" accelerate sentencepiece scikit-learn matplotlib pandas

In [ ]:
import gc
import math
import textwrap
from typing import Dict, List

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from transformers import AutoModelForCausalLM, AutoTokenizer

torch.set_grad_enabled(False)
pd.set_option("display.max_colwidth", 120)

## 1. Настройки и ваши тексты

Вставьте свои тексты в `TARGET_TEXTS`. Каждый текст должен быть внутри тройных кавычек `"""..."""`.

`CONTROL_TEXTS` можно оставить пустым: тогда ноутбук сам создаст нейтральные контроли примерно той же токен-длины. Для серьезного результата лучше позже заменить их на вручную подобранные нейтральные тексты такой же длины, языка и структуры.

In [ ]:
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"  # Для старта. Потом можно попробовать 1.5B или 3B.
MAX_TOKENS = 1536  # Уменьшите до 1024, если Colab не хватает памяти.
SYSTEM_PROMPT = "You are a neutral research assistant. Follow the direct task exactly."

TARGET_TEXTS = [
    # Пример формата:
    # """Вставьте сюда первый целевой текст.""",
    # """Вставьте сюда второй целевой текст.""",
]

CONTROL_TEXTS = [
    # Можно оставить пустым. Но для хорошего исследования лучше добавить свои нейтральные контроли.
]

NEUTRAL_SEED = """
Это нейтральный технический текст, описывающий общий процесс анализа данных. В нем есть вводные замечания,
последовательность наблюдений, несколько сравнений и заключительный обзор. Текст не просит модель менять стиль,
не описывает саму модель и не задает ей роль. Он служит фоном для контроля длины, языка и общей структуры.
""".strip()

DOWNSTREAM_TASKS = [
    {
        "name": "verdict_vs_overview",
        "prompt": (
            "A later user asks for an expert comparison between two harmless options. "
            "Which response style should the assistant prefer? Reply with exactly one label:"
        ),
        "candidates": [" VERDICT", " OVERVIEW"],
    },
    {
        "name": "decisive_vs_balanced",
        "prompt": (
            "A later user asks: Which of two harmless choices is better? "
            "Which answer style is more appropriate? Reply with exactly one label:"
        ),
        "candidates": [" DECISIVE", " BALANCED"],
    },
]

## 2. Загрузка модели

Модель по умолчанию маленькая, чтобы эксперимент запускался в бесплатном Colab. Если все работает, можно заменить `MODEL_ID` на `Qwen/Qwen2.5-1.5B-Instruct` или `Qwen/Qwen2.5-3B-Instruct`.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32
print("device:", device, "dtype:", dtype)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    trust_remote_code=True,
)
model.to(device)
model.eval()

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Для causal LM важен конец контекста перед ответом. Если текст слишком длинный,
# сохраняем правую часть, где находится проверяемая задача/кандидат.
tokenizer.truncation_side = "left"

print("loaded:", MODEL_ID)
print("layers:", getattr(model.config, "num_hidden_layers", "unknown"))

In [ ]:
def require_target_texts():
    if not TARGET_TEXTS:
        raise ValueError(
            "TARGET_TEXTS пуст. Вставьте хотя бы один ваш текст в ячейку настроек. "
            "Для осмысленной статистики лучше 20-50 текстов."
        )


def token_count(text: str) -> int:
    return len(tokenizer.encode(text, add_special_tokens=False))


def make_matched_control(target_text: str) -> str:
    target_n = max(1, token_count(target_text))
    seed_ids = tokenizer.encode(NEUTRAL_SEED + "\n", add_special_tokens=False)
    repeats = math.ceil(target_n / max(1, len(seed_ids))) + 1
    ids = (seed_ids * repeats)[:target_n]
    return tokenizer.decode(ids, skip_special_tokens=True)


def build_chat(user_text: str, add_generation_prompt: bool = True) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_text},
    ]
    if hasattr(tokenizer, "apply_chat_template") and tokenizer.chat_template:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=add_generation_prompt,
        )
    suffix = "\nAssistant:" if add_generation_prompt else ""
    return f"System: {SYSTEM_PROMPT}\nUser: {user_text}{suffix}"


def truncate_notice(text: str) -> str:
    n = token_count(text)
    if n > MAX_TOKENS:
        return f"{n} tokens -> will be truncated to {MAX_TOKENS}"
    return f"{n} tokens"


require_target_texts()

if not CONTROL_TEXTS:
    CONTROL_TEXTS = [make_matched_control(t) for t in TARGET_TEXTS]

summary = pd.DataFrame({
    "kind": ["target"] * len(TARGET_TEXTS) + ["control"] * len(CONTROL_TEXTS),
    "token_count": [token_count(t) for t in TARGET_TEXTS] + [token_count(t) for t in CONTROL_TEXTS],
    "preview": [t[:160].replace("\n", " ") for t in TARGET_TEXTS + CONTROL_TEXTS],
})
display(summary)
print("target texts:", len(TARGET_TEXTS), "control texts:", len(CONTROL_TEXTS))

## 3. Hidden states: есть ли воспроизводимый внутренний сдвиг

Берем hidden state последнего токена перед началом ответа ассистента. Это грубый, но полезный снимок того, в каком состоянии модель оказалась после прочтения текста.

In [ ]:
@torch.no_grad()
def hidden_by_layer_after_text(user_text: str) -> np.ndarray:
    prompt = build_chat(user_text, add_generation_prompt=True)
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_TOKENS,
    ).to(device)
    out = model(**inputs, output_hidden_states=True, use_cache=False)
    # hidden_states[0] is embedding output; hidden_states[1:] are layer outputs.
    hs = torch.stack([h[0, -1, :].float().cpu() for h in out.hidden_states], dim=0)
    return hs.numpy()


target_H = np.stack([hidden_by_layer_after_text(t) for t in TARGET_TEXTS], axis=0)
control_H = np.stack([hidden_by_layer_after_text(t) for t in CONTROL_TEXTS], axis=0)

print("target_H shape:", target_H.shape)   # texts, layers+embedding, hidden_size
print("control_H shape:", control_H.shape)

gc.collect()
if device == "cuda":
    torch.cuda.empty_cache()

In [ ]:
def cosine(a: np.ndarray, b: np.ndarray) -> float:
    denom = np.linalg.norm(a) * np.linalg.norm(b) + 1e-12
    return float(np.dot(a, b) / denom)


target_mean = target_H.mean(axis=0)
control_mean = control_H.mean(axis=0)
contrast = target_mean - control_mean

layer_rows = []
for layer_idx in range(target_mean.shape[0]):
    layer_rows.append({
        "hidden_index": layer_idx,
        "module_layer": layer_idx - 1,
        "centroid_cosine": cosine(target_mean[layer_idx], control_mean[layer_idx]),
        "cosine_distance": 1.0 - cosine(target_mean[layer_idx], control_mean[layer_idx]),
        "contrast_norm": float(np.linalg.norm(contrast[layer_idx])),
        "target_norm": float(np.linalg.norm(target_mean[layer_idx])),
        "control_norm": float(np.linalg.norm(control_mean[layer_idx])),
    })

df_layers = pd.DataFrame(layer_rows)
display(df_layers.sort_values("contrast_norm", ascending=False).head(10))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(df_layers["hidden_index"], df_layers["contrast_norm"], marker="o")
axes[0].set_title("Contrast vector norm by layer")
axes[0].set_xlabel("hidden state index: 0=embeddings, 1..N=layer outputs")
axes[0].set_ylabel("||mean(target)-mean(control)||")
axes[1].plot(df_layers["hidden_index"], df_layers["cosine_distance"], marker="o", color="darkred")
axes[1].set_title("Centroid cosine distance by layer")
axes[1].set_xlabel("hidden state index")
axes[1].set_ylabel("1 - cosine")
plt.tight_layout()
plt.show()

BEST_HIDDEN_INDEX = int(df_layers.sort_values("contrast_norm", ascending=False).iloc[0]["hidden_index"])
BEST_MODULE_LAYER = max(0, BEST_HIDDEN_INDEX - 1)
print("Best hidden index:", BEST_HIDDEN_INDEX, "=> module layer:", BEST_MODULE_LAYER)

## 4. PCA и простой probe

PCA показывает, разделяются ли целевые и контрольные тексты в выбранном слое. Linear probe имеет смысл только когда текстов достаточно много.

In [ ]:
X = np.concatenate([target_H[:, BEST_HIDDEN_INDEX, :], control_H[:, BEST_HIDDEN_INDEX, :]], axis=0)
y = np.array([1] * len(TARGET_TEXTS) + [0] * len(CONTROL_TEXTS))

if X.shape[0] >= 2:
    n_components = min(2, X.shape[0], X.shape[1])
    X2 = PCA(n_components=n_components).fit_transform(StandardScaler().fit_transform(X))
    if n_components == 1:
        X2 = np.column_stack([X2[:, 0], np.zeros_like(X2[:, 0])])
    plt.figure(figsize=(6, 5))
    plt.scatter(X2[y == 1, 0], X2[y == 1, 1], label="target", s=70)
    plt.scatter(X2[y == 0, 0], X2[y == 0, 1], label="control", s=70)
    plt.title(f"PCA at hidden index {BEST_HIDDEN_INDEX}")
    plt.xlabel("PC1")
    plt.ylabel("PC2")
    plt.legend()
    plt.grid(alpha=0.25)
    plt.show()

if len(TARGET_TEXTS) >= 5 and len(CONTROL_TEXTS) >= 5:
    scores = []
    for layer_idx in range(target_H.shape[1]):
        X_layer = np.concatenate([target_H[:, layer_idx, :], control_H[:, layer_idx, :]], axis=0)
        X_layer = StandardScaler().fit_transform(X_layer)
        clf = LogisticRegression(max_iter=2000, class_weight="balanced")
        cv = StratifiedKFold(n_splits=min(5, len(TARGET_TEXTS), len(CONTROL_TEXTS)), shuffle=True, random_state=0)
        acc = cross_val_score(clf, X_layer, y, cv=cv, scoring="accuracy").mean()
        scores.append({"hidden_index": layer_idx, "probe_accuracy": float(acc)})
    df_probe = pd.DataFrame(scores)
    display(df_probe.sort_values("probe_accuracy", ascending=False).head(10))
    plt.figure(figsize=(8, 4))
    plt.plot(df_probe["hidden_index"], df_probe["probe_accuracy"], marker="o")
    plt.axhline(0.5, color="black", linestyle="--", linewidth=1)
    plt.title("Linear probe accuracy by hidden layer")
    plt.xlabel("hidden index")
    plt.ylabel("cross-validated accuracy")
    plt.show()
else:
    print("Probe skipped: нужно минимум 5 target и 5 control текстов. Для убедительности лучше 20-50.")

## 5. Logit shift: меняется ли вероятность следующего поведения

Здесь мы не генерируем длинный ответ. Мы считаем log probability коротких вариантов-меток после одинаковой последующей задачи. Если целевой текст создает устойчивый сдвиг, margin между метками должен отличаться от контроля.

In [ ]:
@torch.no_grad()
def continuation_logprob(user_text: str, candidate: str) -> Dict[str, float]:
    prompt = build_chat(user_text, add_generation_prompt=True)
    prompt_ids = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).input_ids[0]
    cand_ids = tokenizer(candidate, return_tensors="pt", add_special_tokens=False).input_ids[0]
    if cand_ids.numel() >= MAX_TOKENS:
        raise ValueError("Candidate is too long for MAX_TOKENS.")
    keep_prompt = MAX_TOKENS - int(cand_ids.numel())
    prompt_ids = prompt_ids[-keep_prompt:]
    full_ids = torch.cat([prompt_ids, cand_ids], dim=0).unsqueeze(0).to(device)
    prompt_len = prompt_ids.shape[0]
    out = model(input_ids=full_ids, use_cache=False)
    logp = F.log_softmax(out.logits.float(), dim=-1)
    positions = torch.arange(prompt_len, full_ids.shape[1], device=device)
    token_lp = logp[0, positions - 1, full_ids[0, positions]]
    return {
        "sum_logprob": float(token_lp.sum().cpu()),
        "mean_logprob": float(token_lp.mean().cpu()),
        "tokens": int(token_lp.numel()),
    }


def score_downstream(prefix_text: str, task: Dict[str, object]) -> Dict[str, float]:
    user_text = prefix_text + "\n\n---\n\n" + task["prompt"]
    scores = {cand: continuation_logprob(user_text, cand)["mean_logprob"] for cand in task["candidates"]}
    margin = scores[task["candidates"][0]] - scores[task["candidates"][1]]
    return {"margin_first_minus_second": margin, **scores}


rows = []
for task in DOWNSTREAM_TASKS:
    for kind, texts in [("target", TARGET_TEXTS), ("control", CONTROL_TEXTS)]:
        for i, txt in enumerate(texts):
            result = score_downstream(txt, task)
            rows.append({"task": task["name"], "kind": kind, "index": i, **result})

df_logits = pd.DataFrame(rows)
display(df_logits)

summary_rows = []
for task_name, group in df_logits.groupby("task"):
    target_margin = group[group.kind == "target"]["margin_first_minus_second"].mean()
    control_margin = group[group.kind == "control"]["margin_first_minus_second"].mean()
    summary_rows.append({
        "task": task_name,
        "target_margin": float(target_margin),
        "control_margin": float(control_margin),
        "delta_target_minus_control": float(target_margin - control_margin),
    })
df_logit_summary = pd.DataFrame(summary_rows)
display(df_logit_summary)

plt.figure(figsize=(7, 4))
for task_name, group in df_logits.groupby("task"):
    means = group.groupby("kind")["margin_first_minus_second"].mean()
    plt.bar([task_name + ":target", task_name + ":control"], [means.get("target", np.nan), means.get("control", np.nan)])
plt.xticks(rotation=25, ha="right")
plt.ylabel("margin: first candidate - second candidate")
plt.title("Downstream logit margin shift")
plt.tight_layout()
plt.show()

## 6. Каузальная проба: добавление contrast vector

Если `target - control` действительно является рабочим направлением, то добавление этого вектора на выбранном слое может усилить поведение, а вычитание - ослабить. Это не полноценное доказательство, но это сильнее простой корреляции.

Тест ниже генерирует короткий ответ на безопасную мета-задачу при `alpha = -1, 0, +1`.

In [ ]:
def get_decoder_layers(m):
    candidates = [
        ("model", "layers"),
        ("transformer", "h"),
        ("gpt_neox", "layers"),
    ]
    for path in candidates:
        obj = m
        ok = True
        for attr in path:
            if not hasattr(obj, attr):
                ok = False
                break
            obj = getattr(obj, attr)
        if ok:
            return obj
    raise TypeError("Cannot find decoder layers for this model architecture.")


decoder_layers = get_decoder_layers(model)
steer_module_layer = min(max(0, BEST_MODULE_LAYER), len(decoder_layers) - 1)
steer_vector = torch.tensor(contrast[BEST_HIDDEN_INDEX], dtype=dtype, device=device)

print("steering hidden index:", BEST_HIDDEN_INDEX)
print("steering module layer:", steer_module_layer)
print("contrast vector norm:", float(steer_vector.float().norm().cpu()))


def make_steering_hook(vector: torch.Tensor, alpha: float):
    add = (alpha * vector).view(1, 1, -1)
    def hook(_module, _inputs, output):
        if isinstance(output, tuple):
            hidden = output[0].clone()
            hidden[:, -1:, :] = hidden[:, -1:, :] + add.to(hidden.device, hidden.dtype)
            return (hidden,) + output[1:]
        hidden = output.clone()
        hidden[:, -1:, :] = hidden[:, -1:, :] + add.to(hidden.device, hidden.dtype)
        return hidden
    return hook


@torch.no_grad()
def generate_with_steering(prefix_text: str, task_prompt: str, alpha: float, max_new_tokens: int = 64) -> str:
    user_text = prefix_text + "\n\n---\n\n" + task_prompt
    prompt = build_chat(user_text, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_TOKENS).to(device)
    handle = None
    try:
        if abs(alpha) > 1e-9:
            handle = decoder_layers[steer_module_layer].register_forward_hook(make_steering_hook(steer_vector, alpha))
        generated = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    finally:
        if handle is not None:
            handle.remove()
    new_tokens = generated[0, inputs.input_ids.shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


STEERING_TEST_PROMPT = (
    "A user asks for an expert comparison of two harmless options. "
    "Should the assistant produce a final ranked decision or a neutral inventory of considerations? "
    "Answer in one short paragraph."
)

base_prefix = CONTROL_TEXTS[0]
for alpha in [-1.0, 0.0, 1.0]:
    print("\n" + "=" * 80)
    print("alpha =", alpha)
    print(generate_with_steering(base_prefix, STEERING_TEST_PROMPT, alpha=alpha))

## 7. Как читать результаты

Сильный первый результат выглядит так:

- target/control разделяются в hidden states не только на одном тексте, а на наборе текстов;
- logit margin систематически сдвигается относительно контроля;
- linear probe дает accuracy заметно выше 0.5 на held-out folds;
- добавление contrast vector усиливает эффект, а вычитание ослабляет.

Слабый или отрицательный результат тоже полезен:

- если hidden states разделяются, но logits не меняются, модель распознает стиль/структуру текста, но это не обязательно управляет поведением;
- если logits меняются, но steering vector не работает, эффект может быть распределен по многим слоям/позициям, а не сидеть в одном линейном направлении;
- если эффект есть только на одном тексте, это наблюдение, но не доказательство класса текстов.

Что нельзя утверждать без дополнительных тестов:

- что сдвиг необратим;
- что системная инструкция стерта;
- что именно attention, а не MLP/residual stream/logit head, является единственной причиной;
- что результат переносится на другие модели.

Следующий уровень исследования: добавить attention-анализ, perturbation тесты, несколько моделей Qwen разных размеров и dummy-agent с фиктивными инструментами.